In [ ]:
import numpy as p
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving spam.csv to spam.csv


In [ ]:
df = pd.read_csv('spam.csv', encoding='latin1')

In [ ]:
df.head()
df.shape

(5572, 5)

In [ ]:
df.isnull().sum()

,0
v1,0
v2,0
Unnamed: 2,5522
Unnamed: 3,5560
Unnamed: 4,5566


In [ ]:
df = df[['v1','v2']] # Keep only useful columns

In [ ]:
df.columns = ['label','message'] # Rename columns

In [ ]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Convert ham/spam to 0/1
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['label'])

In [ ]:
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
# Features and target
X = df['message']
y = df['label']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Convert text to numerical features
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X = tfidf.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

print("\nModel Comparison")
print("-" * 40)

results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    results[name] = accuracy

    print(f"{name:<20}: {accuracy:.4f}")

# Best model
best_model = max(
    results,
    key=results.get
)

print("\nBest Model:", best_model)
print("Accuracy:", round(results[best_model], 4))

from sklearn.model_selection import cross_val_score

print("\nCross Validation Results")
print("-" * 40)

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='accuracy'
    )

    print(
        f"{name:<20}: "
        f"{scores.mean():.4f}"
    )


Model Comparison
----------------------------------------
Naive Bayes         : 0.9749
Logistic Regression : 0.9489
Linear SVM          : 0.9740
Random Forest       : 0.9776

Best Model: Random Forest
Accuracy: 0.9776

Cross Validation Results
----------------------------------------
Naive Bayes         : 0.9765
Logistic Regression : 0.9582
Linear SVM          : 0.9799
Random Forest       : 0.9785


In [ ]:
rf = RandomForestClassifier()

In [ ]:
cv_scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring='accuracy')

In [ ]:
model = rf.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = rf.predict(X_test)

In [ ]:
# Evaluation
print("\nTest Accuracy:",
      accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Test Accuracy: 0.97847533632287

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       0.99      0.85      0.91       150

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
# Take message from user
message = input("\nEnter a message: ")

# Convert message to TF-IDF features
message_tfidf = tfidf.transform([message])

# Predict
prediction = rf.predict(message_tfidf)

# Display result
if prediction[0] == 1:
    print("\nSpam Message")
else:
    print("\nNot Spam (Ham)")


Enter a message: Free entry in 2 a wkly comp to win FA Cup

Spam Message
